# wald_ols

> 对应代码：`EconometricsCode/code_in_notes/Chap.9/wald_ols.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.9`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.9")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

先清除可能已存在的同名程序，为定义模拟程序做准备。

In [ ]:
%%stata
set seed 19880505
cap program drop dgp

定义模拟程序 dgp：选项 obs 为样本量（默认 1000），b 为真实回归系数（默认 -1）。tempvar 声明临时变量，tempname 声明临时标量与矩阵名，避免命名冲突。

生成数据：x 为标准正态，误差项 u 为正态的三次方——它仍均值为零但具有厚尾、偏离正态，用来检验 Wald 统计量在非正态误差下的表现；y = b·x + u。

做带稳健标准误的 OLS 回归，取出 x 的系数估计 b̂，并从方差协方差矩阵 e(V) 中用 rownumb/colnumb 定位并提取其方差 V_kk。

手工构造非线性假设 H0: β²=1 的 Wald 统计量。由 delta 方法，g(β)=β²−1 的导数为 2β，故统计量为 W=(b̂²−1)²/(4b̂²·V_kk)，渐近服从 χ²(1)。随后计算 p 值：1 减去卡方分布的累积概率。

若 p 值小于 0.05 则记 rejected=1，否则为 0；同时返回检验统计量本身，供后续考察其抽样分布。

In [ ]:
%%stata
program define dgp, rclass
	syntax [, obs(integer 1000) b(real -1.0)]
	drop _all
	set obs `obs'
	tempvar x y u
	tempname vkk V test_stat bk p
	gen `x'=rnormal()
	gen `u'=rnormal()^3
	gen `y'=`b'*`x'+`u'
	reg `y' `x' , robust
	local `bk'=_b[`x']
	mat `V'=e(V)
	local `vkk'=`V'[rownumb(`V',"`x'"),colnumb(`V',"`x'")]
	local `test_stat'=((``bk'')^2-1)^2/(4*(``bk'')^2*``vkk'')
	local `p'=1-chi2(1,``test_stat'')
	if ``p''<0.05{
		return scalar rejected=1
	}
	else{
		return scalar rejected=0
	}
	return scalar teststat=``test_stat''
end

第一组模拟：真实 b=−1，此时 β²=1，原假设成立。重复 2000 次后 su 显示的拒绝频率即检验的实际显著性水平，应接近 5%；hist 画出 Wald 统计量的直方图，可与 χ²(1) 分布比较。

In [ ]:
%%stata
// simulate
simulate rejected=r(rejected) teststat=r(teststat) ,reps(2000):dgp
su
hist teststat

第二组模拟：把真实系数改为 b=0，此时 β²=0≠1，原假设不成立。重复模拟后拒绝频率给出检验功效；同时注意 b̂ 出现在分母中，b 接近 0 时 Wald 统计量行为会恶化，这是非线性假设 Wald 检验的典型病态。

In [ ]:
%%stata
simulate rejected=r(rejected) teststat=r(teststat) ,reps(2000):dgp, b(0)
su